# BSEN41030 — Assignment 2: Scientific LLM-Based Case Study

**Name:** Zhenjiao Du &nbsp;&nbsp; **Dataset:** Umami-peptide-classification &nbsp;&nbsp; **Pre-trained model:** ESM-2 (`facebook/esm2_t6_8M_UR50D`, 320-dimensional embeddings)

Individual assignment · 20% · due Week 9 · independent of the Proposal/Report project

**How to use this template**
- Same structure and standard as Assignment 1, but the method is now a **scientific language model** (e.g. a protein language model) instead of a traditional ML method.
- Complete Sections 1–6 **in order**, then fill in Section 7 (AI acknowledgement).
- A GPU is recommended for larger models or fine-tuning (Colab: **Runtime → Change runtime type → T4 GPU**). Running at your own laptop is also feasible, just take slightly longer time for the representaion part.
- Before submitting, run **Restart & Run All** and save the notebook **with its outputs**.

## 0. Setup

**How to read the code in this notebook**
- A grey box is a **code cell**. Run it with **Shift + Enter**; whatever it prints appears just below it.
- Any text after a `#` is a **comment**: a note for the human reader that Python ignores. The comments explain every step, so you can follow the logic without knowing Python.
- Cells must be run **from top to bottom**, because later cells use results (e.g. `X_train`) created by earlier ones.

The cell below loads the *libraries* (ready-made toolboxes) we need and fixes the random seed.

In [1]:
# If you run this notebook on Google Colab, every library below is already installed.
# On your own computer you may first need to install them once (in a terminal):
#   pip install torch transformers scikit-learn pandas openpyxl

# --- General-purpose libraries ------------------------------------------------
import numpy as np    # "NumPy": tables (arrays) of numbers and the maths on them
import pandas as pd   # "pandas": data tables (DataFrames), like a spreadsheet inside Python

# --- The protein language model -----------------------------------------------
import torch                                       # PyTorch: the engine that runs the ESM-2 neural network
from transformers import AutoTokenizer, AutoModel  # Hugging Face tools that download and run pre-trained models

# --- Traditional machine learning (the "downstream" classifier, as in Assignment 1) ---
from sklearn.pipeline import Pipeline                # chains several steps (scaling -> classifier) into one model
from sklearn.preprocessing import StandardScaler     # rescales every feature to mean 0 and standard deviation 1
from sklearn.linear_model import LogisticRegression  # candidate classifier 1
from sklearn.svm import SVC                          # candidate classifier 2: support vector machine
from sklearn.ensemble import RandomForestClassifier  # candidate classifier 3 (the method of Assignment 1)
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_score  # cross-validation and tuning
from sklearn import metrics                          # evaluation metrics: accuracy, F1, MCC, AUC, ...

RANDOM_SEED = 42                # use this for every random step (split, CV, models, fine-tuning) so your results are reproducible
torch.manual_seed(RANDOM_SEED)  # also fix PyTorch's random numbers (embedding extraction has no randomness, but it is good practice)

# Run ESM-2 on a GPU if there is one (e.g. Colab T4), otherwise on the CPU.
# The smallest ESM-2 model and our short peptides need only a few seconds on a normal laptop CPU.
device = "cuda" if torch.cuda.is_available() else "cpu"
print("ESM-2 will run on:", device)

ESM-2 will run on: cpu


## 1. Data import

**📝 To do**
- Load your dataset into the notebook (dataset list: [BSEN10040-Assignment](https://github.com/dzjxzyd/BSEN10040-Assignment); read the README in your dataset's folder). Use a sequence-based dataset, e.g. peptide or protein sequences for solubility, antimicrobial, antihypertensive, antioxidant, umami or cell-penetrating activity.
- Show its size and the first few rows.
- Briefly describe the dataset: what one sample is, what the input sequence is (type, typical length), what the target is, and whether the task is regression or classification.

In [2]:
# Read the two Excel files supplied with the dataset (they are in the "data" folder next to this notebook).
train_raw = pd.read_excel("data/umami_MRNN_train.xlsx")  # training file
test_raw  = pd.read_excel("data/umami_MRNN_test.xlsx")   # independent test file

# .shape gives (number of rows, number of columns). One row = one peptide.
print("train:", train_raw.shape, "| test:", test_raw.shape)

# Show the first 5 rows of the training table.
train_raw.head()

train: (400, 4) | test: (100, 2)


,sequence,label,Unnamed: 2,Unnamed: 3
0,AE,0,NaN,https://github.com/daibiaoxuwu/umamiMRNN_datasets
1,AH,0,NaN,Umami peptides are labeled as 0 while the Bitt...
2,DA,0,NaN,NaN
3,DD,0,NaN,NaN
4,DE,0,NaN,NaN


In [3]:
# The training file has two extra columns ("Unnamed: 2" and "Unnamed: 3") holding notes from the data provider.
# Print the notes (dropna() skips the empty cells):
for note in train_raw["Unnamed: 3"].dropna():
    print("-", note)

- https://github.com/daibiaoxuwu/umamiMRNN_datasets
- Umami peptides are labeled as 0 while the Bitter peptides are labeled as 1


In [4]:
# How long are the peptides? .str.len() counts the letters (amino acids) in every sequence;
# .describe() summarises those lengths: count, mean, standard deviation, minimum, quartiles and maximum.
all_sequences = pd.concat([train_raw["sequence"], test_raw["sequence"]])  # train + test together
all_sequences.str.len().describe().round(1)

count    500.0
mean       5.5
std        3.9
min        2.0
25%        3.0
50%        4.0
75%        7.0
max       39.0
Name: sequence, dtype: float64

**Dataset description.**

The data come from Qi et al. (2023, *Food Chemistry* 405, 134935), the same dataset as in Assignment 1.

- **One sample** is a short **food-derived peptide**.
- **Input:** the peptide **sequence**, written as one-letter amino-acid codes (e.g. `DE` = aspartic acid + glutamic acid). The sequences are very short: 2–39 amino acids, median 4.
- **Target:** the column `label`, its taste class. According to the provider's note, `0` = **umami** and `1` = **bitter**.
- **Task:** **binary classification** (umami vs bitter).
- **Size:** a training file with 400 peptides and an independent test file with 100 peptides. The training file also contains two note columns, which are not data and will be removed.

The only input is the sequence itself, so it must be turned into numbers before any model can use it. In Assignment 1 we did this by hand (amino-acid composition); here a **protein language model** does it for us (Section 2).

## 2. Preprocessing

**📝 To do** — apply the steps your dataset and chosen model need, for example:
- sequence cleaning (whitespace, upper case, duplicates, non-standard residues such as X, B, Z, U)
- preparing the target if needed (e.g. log-transform, or a justified threshold to create classes)
- encoding for the scientific LLM: tokenisation, or extracting embeddings from a pre-trained protein/chemical language model (e.g. ESM-2, ProtBERT, ChemBERTa)

### 2.1 Sequence cleaning

In [5]:
# Keep only the two useful columns (the sequence and its label) and drop the two note columns.
# .copy() makes an independent table, so later changes do not affect train_raw / test_raw.
train = train_raw[["sequence", "label"]].copy()
test  = test_raw[["sequence", "label"]].copy()

# Tidy the sequences: remove stray spaces at either end (.strip) and make every letter upper case (.upper),
# because ESM-2 only knows upper-case amino-acid letters.
for table in [train, test]:
    table["sequence"] = table["sequence"].str.strip().str.upper()

# Quality checks:
#   missing values     -> empty cells
#   duplicated         -> the same peptide listed twice in one file
#   in both files      -> a test peptide the model would already have seen during training ("data leakage")
for name, table in [("train", train), ("test", test)]:
    print(f"{name}: missing values = {table.isna().sum().sum()}, "
          f"duplicated sequences = {table['sequence'].duplicated().sum()}")
print("sequences in both train and test:", len(set(train["sequence"]) & set(test["sequence"])))

# Non-standard residues: any letter outside the 20 standard amino acids (e.g. X = unknown, B, Z, U).
AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"
letters_used = set("".join(pd.concat([train, test])["sequence"]))  # every distinct letter in the data
print("non-standard letters:", letters_used - set(AMINO_ACIDS) or "none")

train: missing values = 0, duplicated sequences = 0
test: missing values = 0, duplicated sequences = 0
sequences in both train and test: 0
non-standard letters: none


The sequences are already clean: no missing values, no duplicates, no peptide shared between train and test, and only the 20 standard amino acids. The only cleaning step needed is dropping the two note columns.

### 2.2 Preparing the target

The files use `0` = umami and `1` = bitter. We flip this so that **`1` = umami**, because umami is the class we want to find. Metrics such as precision, recall and F1 are calculated for the class coded `1`, so they will now describe how well the model detects **umami** peptides.

In [6]:
# (train["label"] == 0) asks "is this peptide umami?" for every row and gives True/False;
# .astype(int) turns True/False into 1/0. Result: 1 = umami, 0 = bitter.
y_train = (train["label"] == 0).astype(int)
y_test  = (test["label"] == 0).astype(int)

# The mean of a 0/1 column is the fraction of 1s, i.e. the fraction of umami peptides.
print("umami fraction  train: %.3f | test: %.3f" % (y_train.mean(), y_test.mean()))

umami fraction  train: 0.425 | test: 0.420


About 42 % of the peptides are umami in both files, so the classes are only mildly imbalanced.

### 2.3 Encoding with the protein language model ESM-2

**What is ESM-2?** ESM-2 (Lin et al., 2023, *Science*) is a **protein language model** from Meta AI. It is a neural network that was shown tens of millions of natural protein sequences with some amino acids hidden, and learned to guess the hidden ones, much as a text language model learns English by filling in missing words. To do that well, it had to learn which amino acids behave alike and how their neighbours change their role. We reuse that knowledge here **without any further training**.

**Which ESM-2?** We use the smallest version, `esm2_t6_8M_UR50D`: **6** layers, **8 million** parameters, and every amino acid described by **320 numbers**. It runs on a laptop CPU in seconds. (Larger versions give 480, 640, 1280 … numbers per amino acid but are much slower.)

**From a sequence to 320 numbers, in three steps**
1. **Tokenisation.** Each amino-acid letter becomes an integer ID (a *token*). ESM-2 also adds a start token `<cls>` and an end token `<eos>`.
2. **Embedding.** The model outputs a list of 320 numbers (an *embedding*) for **every** token. Each embedding describes that amino acid *in the context of its neighbours*.
3. **Mean pooling.** Peptides have different lengths, but a classifier needs the same number of features for every sample. We therefore **average** the embeddings of the real amino acids (leaving out `<cls>` and `<eos>`), giving **one 320-number vector per peptide**.

These 320 numbers replace the 20 hand-designed amino-acid composition features of Assignment 1.

⚠️ *Is it safe to compute embeddings for the test peptides now?* Yes. ESM-2 is used **frozen**: its weights never change and it never sees our labels, so nothing is learned from the test set. It is like looking each peptide up in a fixed dictionary.

In [7]:
MODEL_NAME = "facebook/esm2_t6_8M_UR50D"  # smallest ESM-2: 6 layers, 8 million parameters, 320 numbers per amino acid

# The tokenizer converts a string of amino-acid letters into the integer IDs the model understands.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Download the pre-trained network (only the first time, ~30 MB) and load it.
# add_pooling_layer=False: we do our own averaging (step 3), so we skip an extra layer that was never trained.
esm_model = AutoModel.from_pretrained(MODEL_NAME, add_pooling_layer=False)
esm_model = esm_model.to(device)  # put the model on the GPU or CPU chosen in Section 0
esm_model.eval()                  # "evaluation mode": we only USE the model, we do not train it

# Count all the numbers (parameters) inside the network. The name says "8M"; the part we load is slightly smaller
# because the final amino-acid-guessing layer, which we do not need, is left out.
n_parameters = sum(p.numel() for p in esm_model.parameters())
print(f"Loaded {MODEL_NAME}: {n_parameters / 1e6:.1f} million parameters, "
      f"{esm_model.config.hidden_size} numbers per amino acid")

Loaded facebook/esm2_t6_8M_UR50D: 7.4 million parameters, 320 numbers per amino acid


In [8]:
# Step 1 (tokenisation) for one example: the first training peptide, "AE".
example = tokenizer("AE")
print("token IDs:", example["input_ids"])                                   # the integers given to the model
print("tokens:   ", tokenizer.convert_ids_to_tokens(example["input_ids"]))  # the same IDs translated back to text

token IDs: [0, 5, 9, 2]
tokens:    ['<cls>', 'A', 'E', '<eos>']


`AE` becomes four tokens: the start token `<cls>`, the two amino acids `A` and `E`, and the end token `<eos>`.

The function below performs steps 1–3 for one peptide. A *function* is a named recipe: we write it once with `def`, then reuse it for all 500 peptides.

In [9]:
def esm_embedding(sequence):
    # Return one vector of 320 numbers describing a single peptide.

    # Step 1 - tokenise: letters -> integer IDs (with <cls> and <eos> added), as PyTorch tensors ("pt").
    tokens = tokenizer(sequence, return_tensors="pt").to(device)

    # Step 2 - run the network. torch.no_grad() says "we are not training", which saves memory and time.
    with torch.no_grad():
        output = esm_model(**tokens)

    # output.last_hidden_state holds the embeddings from the last (6th) layer.
    # Its shape is (1 peptide, number of tokens, 320); [0] selects our single peptide -> (number of tokens, 320).
    per_token = output.last_hidden_state[0]

    # Remove the first token (<cls>) and the last token (<eos>): [1:-1] means "from the 2nd to the second-last".
    per_amino_acid = per_token[1:-1]

    # Step 3 - mean pooling: average over the amino acids (dim=0 = down the rows) -> 320 numbers.
    peptide_vector = per_amino_acid.mean(dim=0)

    # Convert from a PyTorch tensor to a plain Python list of 320 numbers.
    return peptide_vector.cpu().tolist()

In [10]:
# Apply the function to every peptide. "[... for seq in ...]" means: do this for each sequence and collect
# the results in a list. np.array(...) stacks the 320-number lists into a table: one row per peptide.
X_train = np.array([esm_embedding(seq) for seq in train["sequence"]])
X_test  = np.array([esm_embedding(seq) for seq in test["sequence"]])

print("X_train:", X_train.shape, "| X_test:", X_test.shape)  # (number of peptides, 320)

# Peek at the first 3 training peptides and the first 6 of their 320 numbers.
pd.DataFrame(X_train[:3, :6], index=train["sequence"][:3],
             columns=[f"dim {i}" for i in range(1, 7)]).round(3)

X_train: (400, 320) | X_test: (100, 320)


,dim 1,dim 2,dim 3,dim 4,dim 5,dim 6
sequence,,,,,,
AE,0.170,-0.293,0.446,0.167,0.031,0.001
AH,0.114,-0.307,0.207,0.072,-0.146,-0.074
DA,0.096,-0.186,0.373,0.050,-0.193,0.102


Every peptide, whatever its length, is now described by 320 numbers. Unlike the AAC features of Assignment 1, the individual numbers have no simple meaning (`dim 1` is not "fraction of alanine"); the information is spread across all 320 of them.

### 2.4 Scaling

In [11]:
# Compare the 320 features: their averages (mean) and spreads (std) over the training peptides.
# axis=0 means "calculate down each column", i.e. one value per feature.
feature_means = X_train.mean(axis=0)
feature_stds  = X_train.std(axis=0)
print("feature means range from %.2f to %.2f" % (feature_means.min(), feature_means.max()))
print("feature stds  range from %.2f to %.2f" % (feature_stds.min(), feature_stds.max()))

feature means range from -5.04 to 0.49
feature stds  range from 0.06 to 0.27


The 320 features sit on quite different scales. Logistic regression and SVMs are sensitive to this (a feature with a large spread would dominate), so we **standardise** each feature to mean 0 and standard deviation 1 with `StandardScaler`.

The scaler *learns* a mean and a standard deviation from data, so it must be fitted on **training data only**. We therefore do not apply it here; we put it **inside a `Pipeline`** together with the classifier (Section 4). During cross-validation the pipeline then re-fits the scaler on the training folds only, and for the final test it uses the values learned from the training set.

## 3. Train/test split

**📝 To do**
- Split the data into a training set and a held-out test set, with a fixed random seed so the split is reproducible. If your dataset comes with its own test file, use it.
- Report the size of each set.

⚠️ From now on, the test set is used **only** in Section 6. Anything that learns from the data (e.g. a scaler, the downstream model, or fine-tuning) must be fitted on the training set only.

In [12]:
# The dataset comes with its own independent test file, so we keep that split (as the README requires)
# instead of re-splitting at random. The split is therefore identical on every run.
# y_train.sum() counts the 1s (umami); (1 - y_train).sum() counts the 0s (bitter).
print(f"Training set: {len(X_train)} peptides ({y_train.sum()} umami, {(1 - y_train).sum()} bitter)")
print(f"Test set:     {len(X_test)} peptides ({y_test.sum()} umami, {(1 - y_test).sum()} bitter)")

Training set: 400 peptides (170 umami, 230 bitter)
Test set:     100 peptides (42 umami, 58 bitter)


## 4. Model selection

**📝 To do**
- Choose a scientific LLM-based approach covered in class, for example:
  - pre-trained embeddings + a downstream classifier/regressor, or
  - fine-tuning the pre-trained model on your task.

**Approach: frozen ESM-2 embeddings + a downstream classifier.**

ESM-2 turns each peptide into 320 numbers (Section 2.3); a traditional classifier then learns from those numbers to tell umami from bitter. We do **not fine-tune** ESM-2 (i.e. we do not update its 8 million weights) because:
- we have only **400** training peptides, far too few to safely adjust 8 million weights: the model would easily memorise the training set (overfitting);
- fine-tuning needs a GPU and many extra settings to tune, whereas frozen embeddings are computed once in seconds;
- frozen embeddings + a simple classifier is the standard first approach for small biological datasets.

**Which downstream classifier?** We compare three classifiers from class, each with its default settings, using **5-fold stratified cross-validation on the training set only**:
- the 400 training peptides are split into 5 equal parts (*folds*) that keep the umami/bitter ratio (*stratified*);
- each model is trained on 4 folds and scored on the 5th, five times in turn;
- the average of the 5 scores estimates how well the model works on peptides it has not seen.

The score is **F1 for the umami class** (as in Assignment 1), which balances precision and recall.

In [13]:
# The cross-validation plan: 5 folds, shuffled with our fixed seed so the folds are the same on every run.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

# Each candidate is a Pipeline of two steps: "scaler" (standardise the 320 features), then "clf" (the classifier).
candidates = {
    "Logistic regression": Pipeline([("scaler", StandardScaler()),
                                     ("clf", LogisticRegression(max_iter=5000, random_state=RANDOM_SEED))]),
    "SVM (RBF kernel)":    Pipeline([("scaler", StandardScaler()),
                                     ("clf", SVC(random_state=RANDOM_SEED))]),
    # Scaling makes no difference to a random forest; it is kept so that all three use the same recipe.
    "Random forest":       Pipeline([("scaler", StandardScaler()),
                                     ("clf", RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=-1))]),
}

# Cross-validate each candidate and collect the results in a small table.
rows = []
for name, model in candidates.items():
    f1_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1")  # 5 F1 scores, one per fold
    rows.append({"downstream classifier": name,
                 "CV F1 (mean)": f1_scores.mean(),
                 "CV F1 (std)":  f1_scores.std()})

comparison = pd.DataFrame(rows).round(3)
comparison

,downstream classifier,CV F1 (mean),CV F1 (std)
0,Logistic regression,0.813,0.032
1,SVM (RBF kernel),0.834,0.061
2,Random forest,0.809,0.056


With default settings, the **SVM** has the highest cross-validated F1 (0.834 ± 0.061), ahead of logistic regression (0.813 ± 0.032) and the random forest (0.809 ± 0.056). The differences are small compared with the fold-to-fold spread (std), so all three are reasonable choices; we take the SVM forward because it scored best.

**Chosen model: ESM-2 embeddings → `StandardScaler` → SVM with an RBF kernel.**

*What does an SVM do?* Picture every peptide as a point in a 320-dimensional space. A **support vector machine** looks for the boundary that separates the umami points from the bitter points with the **widest possible gap (margin)**. The **RBF kernel** lets that boundary curve, so it can follow more complex shapes than a straight line (or flat plane).

## 5. Hyperparameter optimisation

**📝 To do**
- Tune the most important hyperparameters of the downstream model and/or the fine-tuning settings (e.g. learning rate, number of epochs, batch size), **using the training set only** (cross-validation or a validation split).
- Report the best hyperparameters and their validation score.

ESM-2 itself is frozen, so the hyperparameters to tune are those of the downstream SVM:
- **`C`**: how strongly the SVM is penalised for misclassifying training peptides. Small `C` → smoother boundary that tolerates some mistakes (may under-fit); large `C` → boundary that bends to classify every training peptide correctly (may over-fit).
- **`gamma`**: how far the influence of one training peptide reaches. Small `gamma` → smooth, broad boundary; large `gamma` → wiggly boundary that can memorise individual peptides. The default `"scale"` sets `gamma` = 1 / (number of features × feature variance), which is ≈ 1/320 ≈ 0.003 after standardisation.

We run a **grid search**: every combination of 4 values of `C` × 4 values of `gamma` (16 combinations, including the default `C = 1`, `gamma = "scale"`) is scored with the same 5-fold CV on the **training set only**, again by F1 for umami.

In [14]:
svm_pipeline = candidates["SVM (RBF kernel)"]  # the pipeline chosen in Section 4

# The values to try. "clf__C" means "the parameter C of the pipeline step named clf" (two underscores).
param_grid = {
    "clf__C":     [0.1, 1, 10, 100],
    "clf__gamma": ["scale", 0.0001, 0.001, 0.01],
}

# GridSearchCV tries all 16 combinations x 5 folds = 80 model fits and remembers the best combination.
# n_jobs=-1 uses all CPU cores; return_train_score=True also records the score on the training folds.
search = GridSearchCV(svm_pipeline, param_grid, scoring="f1", cv=cv, n_jobs=-1, return_train_score=True)
search.fit(X_train, y_train)

best_std = search.cv_results_["std_test_score"][search.best_index_]  # spread of the best model's 5 fold scores
print("Best hyperparameters:", search.best_params_)
print(f"Best CV F1: {search.best_score_:.3f} ± {best_std:.3f}")

Best hyperparameters: {'clf__C': 1, 'clf__gamma': 'scale'}
Best CV F1: 0.834 ± 0.061


In [15]:
# The CV results for every combination, as a table (one row per combination).
# "train F1" = score on the 4 folds used for training; "CV F1" = score on the held-out 5th fold (what matters).
results = pd.DataFrame(search.cv_results_)
table = (results[["param_clf__C", "param_clf__gamma", "mean_train_score", "mean_test_score", "std_test_score"]]
         .rename(columns={"param_clf__C": "C", "param_clf__gamma": "gamma",
                          "mean_train_score": "train F1", "mean_test_score": "CV F1",
                          "std_test_score": "CV F1 std"})
         .round(3))
table

,C,gamma,train F1,CV F1,CV F1 std
0,0.1,scale,0.702,0.664,0.046
1,0.1,0.0001,0.000,0.000,0.000
2,0.1,0.001,0.651,0.626,0.050
3,0.1,0.01,0.000,0.000,0.000
4,1.0,scale,0.932,0.834,0.061
5,1.0,0.0001,0.743,0.733,0.016
6,1.0,0.001,0.892,0.831,0.039
7,1.0,0.01,0.992,0.789,0.058
8,10.0,scale,0.999,0.814,0.028
9,10.0,0.0001,0.878,0.829,0.040


The best combination is the default, **`C = 1`, `gamma = "scale"`**, with a CV F1 of **0.834 ± 0.061**. `gamma = 0.001` with `C` = 1, 10 or 100 is practically as good (0.831–0.833). What the table shows:
- **Large `gamma` (0.01) over-fits:** the training F1 is 0.99–1.00 (the model memorises the training peptides), but the CV F1 drops to about 0.79.
- **Large `C` also over-fits:** it pushes the training F1 towards 1.00 without improving the CV F1.
- **Too little flexibility under-fits:** with `C = 0.1` the model is so constrained that for some `gamma` values it predicts *every* peptide as bitter. It then never finds an umami peptide, so F1 = 0.

Tuning did not beat the default here, which is a valid result: it confirms that the default SVM settings already suit these embeddings. The final model is the SVM with `C = 1`, `gamma = "scale"`.

## 6. Evaluation

**📝 To do**
- Evaluate the final (tuned) model on the **held-out test set only** and report the model performance.
- Use evaluation metrics appropriate to your task:
  - regression: e.g. RMSE, R²
  - classification: e.g. accuracy, F1, MCC, AUC

We now use the test set for the first and only time. The metrics, with umami as the positive class:
- **accuracy**: fraction of test peptides classified correctly;
- **precision (umami)**: of the peptides the model *calls* umami, the fraction that really are umami;
- **recall (umami)**: of the peptides that really *are* umami, the fraction the model finds;
- **F1 (umami)**: a single score that balances precision and recall (their harmonic mean);
- **MCC** (Matthews correlation coefficient): from −1 to 1, where 0 = no better than guessing; it stays fair when the classes are imbalanced;
- **ROC AUC**: the chance that a randomly chosen umami peptide gets a higher "umami score" than a randomly chosen bitter one (0.5 = guessing, 1 = perfect).

In [16]:
best_model = search.best_estimator_  # best pipeline (scaler + SVM), already re-fitted by GridSearchCV on the full training set

y_pred  = best_model.predict(X_test)            # predicted class for each test peptide: 1 = umami, 0 = bitter
y_score = best_model.decision_function(X_test)  # the SVM's "umami score": the larger, the more umami-like (used for AUC)

test_scores = pd.Series({
    "accuracy":          metrics.accuracy_score(y_test, y_pred),
    "precision (umami)": metrics.precision_score(y_test, y_pred),
    "recall (umami)":    metrics.recall_score(y_test, y_pred),
    "F1 (umami)":        metrics.f1_score(y_test, y_pred),
    "MCC":               metrics.matthews_corrcoef(y_test, y_pred),
    "ROC AUC":           metrics.roc_auc_score(y_test, y_score),
}, name="test set").round(3)
test_scores.to_frame()

,test set
accuracy,0.860
precision (umami),0.938
recall (umami),0.714
F1 (umami),0.811
MCC,0.719
ROC AUC,0.963


In [17]:
# Confusion matrix: how many test peptides of each true class ended up in each predicted class.
# labels=[1, 0] puts umami first. Correct predictions are on the diagonal (top-left and bottom-right).
cm = metrics.confusion_matrix(y_test, y_pred, labels=[1, 0])
pd.DataFrame(cm, index=["true umami", "true bitter"], columns=["predicted umami", "predicted bitter"])

,predicted umami,predicted bitter
true umami,30,12
true bitter,2,56


**Test-set performance.** The tuned model reaches an accuracy of **0.860**, F1 (umami) of **0.811**, MCC of **0.719** and ROC AUC of **0.963**. The test F1 (0.811) is close to the cross-validation estimate (0.834), so cross-validation gave a fair picture of how the model performs on new peptides.

The confusion matrix shows that the model is **cautious about calling a peptide umami**. When it says umami it is almost always right (precision 0.938; only 2 of the 58 bitter peptides were called umami), but it misses 12 of the 42 umami peptides (recall 0.714). The high AUC (0.963) means the SVM's umami score ranks the peptides very well; most errors come from where the cut-off between the two classes falls, not from a poor ranking.

**Comparison with Assignment 1.** On the same test set, the random forest on amino-acid composition (Assignment 1) reached accuracy 0.900, MCC 0.797 and ROC AUC 0.960. These three metrics do not depend on which class is called "positive", so they can be compared directly (F1, precision and recall cannot, because Assignment 1 used the original coding with label 1 as the positive class). The ESM-2 model ranks peptides as well as Assignment 1 (AUC 0.963 vs 0.960) but classifies slightly fewer of them correctly. A likely reason is that ESM-2 learned from full-length proteins, typically hundreds of amino acids long, whereas our peptides have a median length of only 4: its context-based knowledge has little context to work with, while for such short peptides simple composition is already very informative. Possible next steps are a larger ESM-2 model (e.g. `esm2_t12_35M_UR50D`, 480 numbers per amino acid), fine-tuning, or combining ESM-2 embeddings with AAC features.

## 7. AI acknowledgement

**📝 To do**
- State which AI-assisted tools you used (if any) and what you used them for. If none, write "No AI tools were used." You remain responsible for understanding and explaining every step of this notebook.

I used Claude Code (Claude Opus 5.5) to help me write and annotate the code section by section, following the experimental plan I proposed. I checked every step and can explain it.